# 01. LLM-as-Judge

用 LLM 来评估 LLM Agent 的输出——这听起来像循环论证，但实际上是目前最常用的评估方法。关键在于：**如何设计 rubric 和降低 bias**。

> **检查点**：能说出 LLM-as-Judge 的三大 bias（position、verbosity、self-enhancement）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 模拟 LLM-as-Judge：用规则引擎模拟 rubric-based 评分
from dataclasses import dataclass, field

@dataclass
class Rubric:
    dimensions: list[dict]  # [{name, weight, criteria}]

@dataclass
class JudgeResult:
    scores: dict; total: float; reasoning: str

class ToyJudge:
    """基于规则打分的 toy judge——不调用真实 LLM"""
    def __init__(self, rubric: Rubric):
        self.rubric = rubric
    
    def evaluate(self, response: str, question: str) -> JudgeResult:
        scores = {}
        reasons = []
        for dim in self.rubric.dimensions:
            name = dim["name"]
            if name == "accuracy":
                keywords = dim.get("keywords", [])
                hits = sum(1 for kw in keywords if kw.lower() in response.lower())
                score = min(hits / max(len(keywords), 1), 1.0)
            elif name == "completeness":
                score = min(len(response.split()) / dim.get("min_words", 20), 1.0)
            elif name == "clarity":
                score = 1.0 if len(response) < 200 else 0.7
            else:
                score = 0.5
            scores[name] = score
            reasons.append(f"{name}: {score:.2f}")
        
        total = sum(scores[d["name"]] * d.get("weight", 0.33) for d in self.rubric.dimensions)
        return JudgeResult(scores, total, "; ".join(reasons))

rubric = Rubric([
    {"name":"accuracy","weight":0.5,"keywords":["python","guido","1991"]},
    {"name":"completeness","weight":0.3,"min_words":15},
    {"name":"clarity","weight":0.2},
])

judge = ToyJudge(rubric)

good = "Python was created by Guido van Rossum in 1991. It is a high-level language."
bad = "Python is a language."

for label, resp in [("Good",good),("Bad",bad)]:
    result = judge.evaluate(resp, "What is Python?")
    print(f"{label}: score={result.total:.2f}")
    for dim, s in result.scores.items():
        print(f"  {dim}: {s:.2f}")

Good: score=0.98
  accuracy: 1.00
  completeness: 0.93
  clarity: 1.00
Bad: score=0.45
  accuracy: 0.33
  completeness: 0.27
  clarity: 1.00


## Position Bias 实验

LLM-as-Judge 的一个已知问题是 **position bias**：先看到的回答更容易得高分。

在真实系统中，缓解策略包括：
- 交换顺序评两次
- 用多个 judge 独立评分
- 用锚定样本校准

> **检查点**：能设计一个实验来量化你的 judge 的 position bias。

## 真实 API 实验

Position bias 实测：同一对答案交换顺序让真实 judge 评两次，观察分数是否翻转。


In [3]:
# Position bias 实验（真实 judge）
good = "TCP 通过序列号与确认机制保证可靠传输：每个字节都有序号，接收方累计确认，丢失则重传。"
bad = "TCP 很可靠因为它很靠谱，大家都用它所以可靠。"

def judge(a, b):
    out = chat_json(client, f"""哪个回答更好？
A: {a}
B: {b}
返回 JSON: {{"winner": "A或B", "reason": "一句话"}}""")
    return out.get("winner"), out.get("reason")

w1, r1 = judge(good, bad)
w2, r2 = judge(bad, good)  # 交换顺序
print(f"顺序 1（好在 A）: winner={w1} — {r1}")
print(f"顺序 2（好在 B）: winner={w2} — {r2}")
consistent = (w1 == "A" and w2 == "B")
print(f"\njudge 一致性: {'✅ 通过' if consistent else '⚠️ 暴露了 position bias！两次 winner 应指向同一内容'}")


顺序 1（好在 A）: winner=A — A 正确解释了 TCP 通过序列号和确认机制实现可靠传输，而 B 是循环论证且缺乏技术细节。
顺序 2（好在 B）: winner=B — B准确解释了TCP可靠性的具体机制，而A是循环论证且缺乏实质内容

judge 一致性: ✅ 通过


## 练习

1. 扩展 ToyJudge：增加 pairwise comparison 模式（比较两个回答，选更好的）。
2. 实现 multi-judge aggregation：3 个 judge 独立评分，取中位数或加权平均。
3. 思考：如果一个 judge 对所有回答都打高分（leniency bias），如何检测和校准？